# B3 — mixed displacement-stress formulation: TWO-STAGE curriculum (attempt 3)

**Follow-up paper, Professor Rabczuk's item 1.** Attempt 1 (`p_output_scale=500.0`, full 50,000 iterations) and attempt 2 (`p_output_scale=250.0`, recalibrated via `calibrate_p_output_scale.py`) **both failed in the same way**: the `constitutive` loss term oscillated with no downward trend throughout training (attempt 2 was interrupted at iteration 14,000/50,000 once this became clear — identical oscillation, same magnitude range, as attempt 1 showed across its full run). `equilibrium` converges to ~0 within the first ~500 iterations in both attempts and stays there. **Conclusion: the scale correction was real but was never the dominant problem.**

**New hypothesis**: jointly learning u and P from a random initialization, under a loss with no energy term at all, is a much harder optimization landscape than `train_B3.py`'s own single, already-proven energy objective. This run tests a two-stage curriculum instead of changing the loss formula again: **Stage 1** (iterations 1–10,000) trains u ALONE via `train_B3.py`'s own `total_potential_energy_B3` (exactly the baseline's proven objective; the stress head gets no gradient yet and stays at its random init). **Stage 2** (iterations 10,001–50,000) switches to the normal `mixed_stress_loss_B3` objective (constitutive + equilibrium), continuing from the SAME model weights — so the harder joint objective now starts from an already-good displacement field instead of a random one.

**Honest caveat**: 10,000 pretraining iterations is a reasonable starting split within the existing 50,000-iteration budget, not a value derived from a convergence-curve measurement of the baseline (that curve was not re-examined for this specific split). Watch Stage 1's own printed `loss(mean U)` — it should settle to a stable plateau well before iteration 10,000, the same shape `train_B3.py`'s own training always shows; if it is still trending down hard at iteration 10,000, interrupt and report rather than moving on to Stage 2 from an undertrained u. Once Stage 2 begins, watch `constitutive`: if it shows the same flat oscillation (no trend) that doomed attempts 1 and 2, interrupt and report rather than running the rest unattended — this is now the second independent fix being tested, not a confirmed solution.

**Writes to a NEW output directory** (`b3_mixed_stress_two_stage`), not `b3_mixed_stress_training_v2` (attempt 2's own interrupted, incomplete run) — keeps all three attempts' checkpoints and logs separately inspectable.

In [ ]:
# =====================================================================
#  CELL -- B3 mixed displacement-stress: TWO-STAGE curriculum (10,000
#  iterations pretraining u alone via the proven energy objective, then
#  40,000 iterations of the full mixed-stress objective), then
#  evaluation against the held-out FEM dataset.
#  See the markdown cell above for why attempts 1 and 2 failed and what
#  this attempt changes.
# =====================================================================
import os
os.environ['JAX_PLATFORMS'] = 'cpu'

import subprocess
import sys
import time

_started = time.time()


def run(cmd):
    print('$', ' '.join(str(c) for c in cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    p.wait()
    if p.returncode != 0:
        raise subprocess.CalledProcessError(p.returncode, cmd)


from google.colab import drive
if os.path.isdir('/content/drive/MyDrive'):
    print('Drive already mounted at /content/drive.')
else:
    drive.mount('/content/drive')

REPO = '/content/OMAR'
if not os.path.isdir(REPO):
    run(['git', 'clone', '-b', 'claude/claude-code-question-d307wp',
         'https://github.com/SUHIBAMRO/OMAR.git', REPO])
else:
    run(['git', '-C', REPO, 'fetch', 'origin', 'claude/claude-code-question-d307wp'])
    run(['git', '-C', REPO, 'checkout', 'claude/claude-code-question-d307wp'])
    run(['git', '-C', REPO, 'reset', '--hard', 'origin/claude/claude-code-question-d307wp'])

run([sys.executable, '-m', 'pip', 'install', '-q', 'pyvista<0.49'])
run([sys.executable, '-m', 'pip', 'install', '-q', 'torch-fem'])

WORK = f'{REPO}/Practical_Examples'
os.chdir(WORK)
sys.path.insert(0, WORK)

for _mod_name in list(sys.modules):
    if (_mod_name == 'omar_pfem' or _mod_name.startswith('omar_pfem.')
            or _mod_name == 'torchfem' or _mod_name.startswith('torchfem.')
            or _mod_name == 'pyvista' or _mod_name.startswith('pyvista.')):
        del sys.modules[_mod_name]

import torch
assert torch.cuda.is_available(), 'this cell needs a real GPU'
print('GPU:', torch.cuda.get_device_name(0))

import gc
for _attr in ('last_traceback', 'last_value', 'last_type'):
    if hasattr(sys, _attr):
        delattr(sys, _attr)
gc.collect()
torch.cuda.empty_cache()
print(f'GPU memory at start: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated, '
      f'{torch.cuda.memory_reserved()/1e9:.2f} GB reserved (should be ~0 -- if not, '
      'use Runtime > Restart session before trusting this run)')

from omar_pfem.train_B3_mixed_stress import get_args, train
from omar_pfem.data.data_generate_B3_dataset import DEFAULT_RESOLUTION

device = torch.device('cuda')

R = '/content/drive/MyDrive/pfem_run'
OUTPUT_DIR = f'{R}/b3_mixed_stress_two_stage'
os.makedirs(OUTPUT_DIR, exist_ok=True)

args = get_args([
    '--n_iters', '50000',
    '--pretrain_iters', '10000',
    '--batch_size', '8',
    '--log_every', '500',
    '--ckpt_every', '5000',
    '--output_dir', OUTPUT_DIR,
    # p_output_scale=250.0, w_equilibrium=28000 are this script's own
    # defaults (recalibrated attempt 2's value) -- not overridden here;
    # this attempt changes the CURRICULUM, not the scale/weights again.
])
n_elem = (DEFAULT_RESOLUTION[0] - 1) * (DEFAULT_RESOLUTION[1] - 1) * (DEFAULT_RESOLUTION[2] - 1)
print(f'\nTraining at {DEFAULT_RESOLUTION} -> {n_elem} elements, {args.n_iters} iterations '
      f'({args.pretrain_iters} pretrain-u + {args.n_iters - args.pretrain_iters} mixed-stress), '
      f'batch_size={args.batch_size}, w_constitutive={args.w_constitutive}, '
      f'w_equilibrium={args.w_equilibrium}, p_output_scale={args.p_output_scale}, '
      f'checkpoints -> {OUTPUT_DIR}')
print('Stage 1 (pretrain-u): watch loss(mean U) settle to a plateau before iteration '
      f'{args.pretrain_iters} -- if it is still trending down hard right at the stage boundary, '
      'interrupt and report rather than moving on with an undertrained u.')
print('Stage 2 (mixed-stress): watch constitutive -- if it shows the same flat, no-trend '
      'oscillation that doomed attempts 1 (scale=500) and 2 (scale=250), interrupt and report.')

torch.cuda.reset_peak_memory_stats(device)
t0 = time.time()
model = train(args, device)
elapsed_total = time.time() - t0

print(f'\n{"=" * 90}')
print(f'Training finished in {elapsed_total:.1f}s ({elapsed_total / args.n_iters:.4f}s/iteration average).')
print(f'GPU peak memory during training: '
      f'{torch.cuda.max_memory_allocated(device) / 1e6:.1f}MB allocated, '
      f'{torch.cuda.max_memory_reserved(device) / 1e6:.1f}MB reserved.')

# =====================================================================
#  Evaluation: same held-out FEM dataset the baseline DEM checkpoint was
#  evaluated on, so the two sets of numbers are directly comparable.
# =====================================================================
print(f'\n{"=" * 90}')
print('=== Evaluating the final checkpoint against the held-out FEM dataset ===')
CHECKPOINT = f'{OUTPUT_DIR}/checkpoint_{args.n_iters}.pt'
DATASET = f'{R}/b3_dataset_clean_holdout/dataset.h5'
OUT_JSON = f'{OUTPUT_DIR}/qois_{args.n_iters}.json'

sys.argv = [
    'evaluate_B3_mixed_stress.py',
    '--checkpoint', CHECKPOINT,
    '--dataset', DATASET,
    '--out_json', OUT_JSON,
]
from omar_pfem.evaluate_B3_mixed_stress import main as evaluate_main
evaluate_main()

try:
    from omar_pfem.run_manifest import write_manifest
    write_manifest(OUTPUT_DIR, kind='b3_mixed_stress_training',
                    args={'n_iters': args.n_iters, 'pretrain_iters': args.pretrain_iters,
                          'batch_size': args.batch_size,
                          'resolution': DEFAULT_RESOLUTION, 'lr': args.lr,
                          'w_constitutive': args.w_constitutive,
                          'w_equilibrium': args.w_equilibrium,
                          'p_output_scale': args.p_output_scale},
                    started_at=_started,
                    results={'elapsed_total_s': elapsed_total,
                             's_per_iter': elapsed_total / args.n_iters},
                    outputs=[CHECKPOINT, OUT_JSON],
                    notes="Third real training attempt of the mixed displacement-stress "
                          "formulation (Professor Rabczuk's item 1). Attempts 1 "
                          "(p_output_scale=500.0) and 2 (p_output_scale=250.0, recalibrated "
                          "via calibrate_p_output_scale.py) both failed with constitutive "
                          "showing no downward trend -- attempt 2 was interrupted at 14,000/"
                          "50,000 iterations once this became clear, confirming the scale fix "
                          "made no visible difference. This attempt uses a two-stage "
                          "curriculum instead (--pretrain_iters 10000): u alone via the "
                          "proven energy objective first, then the full mixed-stress "
                          "objective continuing from that checkpoint. See "
                          "train_B3_mixed_stress.py's own 'UPDATE 2026-10-21' docstring "
                          "entry for the full diagnosis.")
except Exception as e:
    print(f'[manifest] not recorded: {e}')

print('\nDone. Send this cell\'s full printed output back, especially the "Key comparison '
      'for Professor Rabczuk\'s item 1" line near the end of the evaluation output.')
